<div style="background:linear-gradient(135deg,#0A1628,#0D2440,#102850);
  padding:44px 32px;border-radius:16px;text-align:center;color:#E8F4FD;
  border:1px solid #1E3A6A;box-shadow:0 8px 32px rgba(0,0,0,.4)">
<h1 style="margin:0 0 10px;font-size:2.3em;font-weight:700;letter-spacing:-.5px">
  📐 Demostración Rigurosa de las Ecuaciones de Diseño<br>de Reactores Ideales</h1>
<h2 style="margin:0 0 8px;font-weight:300;opacity:.9;font-size:1.25em">
  Derivaciones completas desde primeros principios · Verificaciones numéricas · Visualización interactiva</h2>
<p style="margin:4px 0;opacity:.8">Fogler — <em>Elements of Chemical Reaction Engineering</em>, 4ª Ed.</p>
<p style="margin:4px 0;opacity:.7;font-size:.9em;font-style:italic">
  Programa 740484 · Maestría en Ingeniería Química · Universidad del Valle<br>
  Prof. José Antonio Lara Ramos, Ing.Qco., M.Sc., Ph.D.</p>
</div>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyArrowPatch, Rectangle, FancyBboxPatch
from matplotlib.lines import Line2D
from scipy.integrate import quad, odeint, solve_ivp
from scipy.optimize import brentq, fsolve
import warnings
warnings.filterwarnings('ignore')

# ── Estilo global ────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor':'#0A0F1E', 'axes.facecolor':'#0A1530',
    'axes.edgecolor':'#2A4080',   'grid.color':'#1A2840',
    'grid.alpha':.35,             'text.color':'#C8D8F0',
    'axes.labelcolor':'#A0C0FF',  'xtick.color':'#8090B0',
    'ytick.color':'#8090B0',      'font.size':11,
    'axes.titlesize':12,          'lines.linewidth':2.2,
    'figure.dpi':110,
})
COLORS = dict(gold='#FFD080', blue='#4A9EFF', green='#2ECC71',
              red='#E74C3C', orange='#FF8C42', purple='#C8A0FF',
              teal='#1BC4C4', white='#E8F4FD', gray='#607090')
print("✅ Librerías cargadas · Versión numpy:", np.__version__)


---
# 0. Fundamento epistemológico: ¿Qué significa "demostración rigurosa"?

En ingeniería química, una ecuación de diseño es **rigurosa** cuando:

1. **Se deriva explícitamente** desde primeros principios (conservación de masa y energía) sin saltarse pasos.
2. **Cada suposición se identifica y justifica** indicando cuándo puede violarse en la práctica.
3. **El resultado se verifica numéricamente** contra soluciones analíticas conocidas o datos experimentales.
4. **La generalidad se establece** mostrando que los casos especiales (isotérmico, adiabático, etc.) emergen como límites del caso general.

## Mapa de derivaciones de este notebook

```
Ley de Conservación de Masa (Postulado universal)
         │
         ├─► Balance molar en sistema abierto  (Ec. General)
         │              │
         │    ┌─────────┴──────────┬──────────────┐
         │    ▼                    ▼              ▼
         │  CSTR                  PFR            BATCH
         │  (mezcla perfecta,     (sin mezcla    (sin flujo,
         │   SS, T uniforme)       axial, SS)     volumen cte.)
         │    │                    │              │
         │    └────────────────────┴──────────────┘
         │                         │
         └─► Balance de Energía ───┴─► Sistemas NO Isotérmicos
                                        (CSTR múltiple SS · PFR hotspot · Batch TMR)
```

> **Referencia canónica:** Fogler, H.S. (2006). *Elements of Chemical Reaction Engineering*, 4ª Ed.
> Pearson. Capítulos 1–9.


---
# 1. REACTOR CSTR — Derivación rigurosa de la ecuación de diseño

## 1.1 Punto de partida: Ley de Conservación de Masa

Para cualquier **especie química A** en cualquier **volumen de control** $\mathcal{V}$,
la ley de conservación de masa en términos molares (sin creación ni destrucción de átomos) establece:

$$\underbrace{\dot{N}_{A,\text{entrada}}}_{\text{flujo molar IN}} - \underbrace{\dot{N}_{A,\text{salida}}}_{\text{flujo molar OUT}} + \underbrace{\int_{\mathcal{V}} r_A \, dV}_{\text{generación por rxn}} = \underbrace{\frac{dN_A}{dt}}_{\text{acumulación}}$$

En forma compacta **(Fogler Ec. 1-4)**:
$$\boxed{F_{A0} - F_A + \int_{\mathcal{V}} r_A \, dV = \frac{dN_A}{dt}}$$

donde $r_A$ [mol m⁻³ s⁻¹] puede depender de la posición dentro de $\mathcal{V}$.

## 1.2 Suposiciones del CSTR ideal — cada una con su consecuencia

| # | Suposición física | Consecuencia matemática | Cuándo falla industrialmente |
|---|------------------|------------------------|------------------------------|
| **S1** | **Mezcla perfecta** — composición y T uniformes en todo $V$ | $r_A(\mathbf{x}) = r_A\big\|_\text{sal} = \text{cte}$ → $\int r_A dV = r_A V$ | $P/V < 0.5$ kW/m³, viscosidad alta, $V > 50$ m³ |
| **S2** | **Estado estacionario** — propiedades independientes del tiempo | $\dfrac{dN_A}{dt} = 0$ | Arranques, paradas, perturbaciones > 10% |
| **S3** | **Isotérmico** — temperatura uniforme y constante | $k = k(T_\text{op}) = \text{cte}$ | $\Delta T_\text{ad} > 30$ K, fallo de camisa |
| **S4** | **Densidad constante** — flujos volumétricos iguales ($v = v_0$) | $C_A = C_{A0}(1-X)$ exacto | Gases con $\delta \neq 0$; reacciones muy exotérmicas |

## 1.3 Derivación algebraica completa — 6 pasos explícitos

**Paso 1:** Aplicar S2 (estado estacionario) a la Ec. general:
$$F_{A0} - F_A + r_A V = 0$$

**Paso 2:** Expresar $F_A$ en términos de la conversión $X$ (definición de $X$):
$$F_A = F_{A0}(1-X) \quad \Rightarrow \quad F_{A0} - F_{A0}(1-X) = F_{A0} X$$

**Paso 3:** Sustituir y despejar $V$, usando $-r_A > 0$ (convenio para reactivos):
$$F_{A0} X = (-r_A)\big|_\text{sal} \cdot V \quad \Rightarrow \quad \boxed{V_\text{CSTR} = \frac{F_{A0} X}{(-r_A)\big|_\text{sal}}}$$

**Paso 4:** Dividir numerador y denominador por $v_0$, y usar $\tau = V/v_0$, $C_{A0} = F_{A0}/v_0$:
$$\tau_\text{CSTR} = \frac{C_{A0} X}{(-r_A)\big|_\text{sal}}$$

**Paso 5:** Sustituir cinética de n-ésimo orden $-r_A = k C_{A0}^n(1-X)^n$:
$$\tau = \frac{X}{k C_{A0}^{n-1}(1-X)^n}$$

**Paso 6:** Definir el **Número de Damköhler** $Da \equiv k\tau C_{A0}^{n-1}$. Para $n=1$:
$$Da = \frac{X}{1-X} \quad \Rightarrow \quad \boxed{X = \frac{Da}{1+Da}} \quad \text{(solución cerrada para 1er orden)}$$

> **Propiedad fundamental:** El CSTR evalúa $(-r_A)$ en la concentración de **salida** — la más baja posible. Consecuencia directa de S1 (mezcla perfecta). Esto implica $V_\text{CSTR} > V_\text{PFR}$ para cualquier $n > 0$.


In [ ]:
# ═══════════════════════════════════════════════════════════
# VERIFICACIÓN NUMÉRICA — Ecuación de diseño CSTR
# Fogler Example 5-1: Hidratación de óxido de propileno
# ═══════════════════════════════════════════════════════════

# Datos del problema
k   = 0.257      # min⁻¹  — cinética 1er orden
CA0 = 2.05       # mol/dm³
v0  = 46.8       # dm³/min
X   = 0.80       # conversión objetivo

FA0 = CA0 * v0   # mol/min

# ── Método 1: Ecuación de diseño directa ────────────────────
rA_sal = k * CA0 * (1 - X)          # mol/(dm³·min)
V_CSTR = FA0 * X / rA_sal            # dm³

# ── Método 2: Verificación por Damköhler ────────────────────
tau    = V_CSTR / v0
Da     = k * tau
X_verif = Da / (1 + Da)

# ── Método 3: Verificación por balance directo ───────────────
FA_out  = FA0 * (1 - X)              # mol/min — flujo salida
rxn_mol = FA0 * X                    # mol/min — convertidos
gen_chk = rA_sal * V_CSTR            # mol/min — generación (negativa)

print("╔══════════════════════════════════════════════════════╗")
print("║  VERIFICACIÓN CSTR — Fogler Example 5-1             ║")
print("╠══════════════════════════════════════════════════════╣")
print(f"║  k         = {k:.3f} min⁻¹                            ║")
print(f"║  CA0       = {CA0:.2f} mol/dm³                         ║")
print(f"║  v0        = {v0:.1f} dm³/min                         ║")
print(f"║  X_des     = {X:.2f}                                  ║")
print("╠══════════════════════════════════════════════════════╣")
print(f"║  FA0       = {FA0:.2f} mol/min                        ║")
print(f"║  (-rA)|sal = {rA_sal:.4f} mol/(dm³·min)               ║")
print(f"║  V_CSTR    = {V_CSTR:.1f} dm³  ← resultado principal  ║")
print(f"║  τ         = {tau:.2f} min                            ║")
print(f"║  Da        = {Da:.4f}                                ║")
print(f"║  X (vía Da) = {X_verif:.4f}  ← verificación ✓        ║")
print("╠══════════════════════════════════════════════════════╣")
print(f"║  Balance: FA0·X = {rxn_mol:.2f} mol/min convertidos  ║")
print(f"║           rA·V  = {gen_chk:.2f} mol/min generados    ║")
print(f"║           Error = {abs(rxn_mol-gen_chk)/rxn_mol*100:.6f}%  ║")
print("╚══════════════════════════════════════════════════════╝")

# ── Análisis de sensibilidad: V vs X ────────────────────────
X_range = np.linspace(0.01, 0.99, 500)
rA_range = k * CA0 * (1 - X_range)
V_range  = FA0 * X_range / rA_range

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('CSTR — Verificación numérica de la ecuación de diseño', color=COLORS['white'], fontweight='bold')

# Panel 1: V vs X
ax = axes[0]
ax.plot(X_range, V_range, color=COLORS['gold'], lw=2.5)
ax.axvline(X, color=COLORS['red'], ls='--', lw=1.8, label=f'X = {X:.2f}')
ax.axhline(V_CSTR, color=COLORS['blue'], ls=':', lw=1.8, label=f'V = {V_CSTR:.0f} dm³')
ax.scatter([X], [V_CSTR], color=COLORS['gold'], s=120, zorder=5)
ax.set_xlabel('Conversión X')
ax.set_ylabel('Volumen CSTR (dm³)')
ax.set_title('V(X) — ecuación de diseño')
ax.legend(); ax.grid(True)
ax.annotate(f'({X:.2f}, {V_CSTR:.0f})', (X, V_CSTR),
            xytext=(X-0.25, V_CSTR+200), color=COLORS['gold'], fontsize=10,
            arrowprops=dict(arrowstyle='->', color=COLORS['gold']))

# Panel 2: X vs Da — verificación algebraica
Da_range = np.linspace(0, 15, 300)
X_Da = Da_range / (1 + Da_range)      # solución analítica 1er orden
ax2 = axes[1]
ax2.plot(Da_range, X_Da, color=COLORS['teal'], lw=2.5, label='X = Da/(1+Da)')
ax2.axvline(Da, color=COLORS['red'], ls='--', lw=1.8, label=f'Da = {Da:.2f}')
ax2.axhline(X, color=COLORS['blue'], ls=':', lw=1.8, label=f'X = {X:.2f}')
ax2.scatter([Da], [X], color=COLORS['gold'], s=120, zorder=5)
ax2.set_xlabel('Número de Damköhler (Da = kτ)')
ax2.set_ylabel('Conversión X')
ax2.set_title('X = Da/(1+Da) — verificación por Da')
ax2.legend(); ax2.grid(True)

plt.tight_layout()
plt.savefig('cstr_verificacion.png', dpi=120, bbox_inches='tight', facecolor='#0A0F1E')
plt.show()
print("✅ Figura guardada: cstr_verificacion.png")


---
# 2. Diagrama de Levenspiel — demostración geométrica de la desigualdad $V_\text{CSTR} > V_\text{PFR}$

## 2.1 El teorema geométrico fundamental

Definimos la **función de Levenspiel** $\mathcal{L}(X)$:
$$\mathcal{L}(X) \equiv \frac{F_{A0}}{-r_A(X)} \quad \left[\text{dm}^3\right]$$

Las ecuaciones de diseño se convierten en áreas geométricas:

$$V_\text{CSTR} = \mathcal{L}(X_\text{sal}) \cdot X_\text{sal} \quad \leftarrow \text{rectángulo de altura } \mathcal{L}(X_\text{sal}) \text{ y base } X_\text{sal}$$

$$V_\text{PFR} = \int_0^{X_A} \mathcal{L}(X)\, dX \quad \leftarrow \text{área bajo la curva } \mathcal{L}(X)$$

## 2.2 Demostración formal de la desigualdad

**Teorema:** Para cualquier cinética con $\dfrac{d(-r_A)}{dX} < 0$ (velocidad decreciente con la conversión) y $n > 0$:

$$V_\text{CSTR} > V_\text{PFR} \quad \forall\, X \in (0,1)$$

**Demostración:** $\mathcal{L}(X)$ es una función monótonamente creciente en $X$ cuando $-r_A$ decrece con $X$.
Para cualquier función creciente $\mathcal{L}$ y $X_\text{sal} \in (0,1)$:

$$\mathcal{L}(X) < \mathcal{L}(X_\text{sal}) \quad \forall\, X < X_\text{sal}$$

Por tanto:
$$V_\text{PFR} = \int_0^{X_\text{sal}} \mathcal{L}(X)\,dX < \int_0^{X_\text{sal}} \mathcal{L}(X_\text{sal})\,dX = \mathcal{L}(X_\text{sal})\cdot X_\text{sal} = V_\text{CSTR}$$

**Q.E.D.** $\square$

## 2.3 Caso especial: reacción autocatalítica

Para $-r_A = k X(1-X)$ (autocatalítica), la velocidad tiene un **máximo** en $X = 0.5$.
$\mathcal{L}(X)$ tiene un **mínimo** en $X = 0.5$ — la curva tiene forma de "U".
En este caso, el CSTR **puede** ser más eficiente que el PFR para ciertas conversiones
(el rectángulo puede quedar completamente bajo la curva en la región del mínimo).


In [ ]:
# ═══════════════════════════════════════════════════════════
# DIAGRAMA DE LEVENSPIEL — Demostración geométrica
# Comparación orden n = 1, 2 y autocatalítica
# ═══════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.suptitle('Diagrama de Levenspiel — Demostración geométrica\n'
             r'$V_\mathrm{CSTR}$ = rectángulo   $|$   $V_\mathrm{PFR}$ = área bajo curva',
             color=COLORS['white'], fontweight='bold', fontsize=12)

casos = [
    ('Cinética 1er orden\n$-r_A = kC_{A0}(1-X)$', 1, False),
    ('Cinética 2do orden\n$-r_A = kC_{A0}^2(1-X)^2$', 2, False),
    ('Autocatalítica\n$-r_A = kXC_{A0}^2(1-X)$', 2, True),
]

k_ev, CA0_ev, v0_ev = 0.5, 2.0, 10.0
FA0_ev = k_ev * CA0_ev * v0_ev   # simplificado
X_des  = 0.80

for ax, (titulo, n, auto) in zip(axes, casos):
    X_arr = np.linspace(0.001, 0.995, 800)

    if auto:
        rA_arr = k_ev * X_arr * CA0_ev**2 * (1 - X_arr)
        # para autocatalítica, necesitamos datos de entrada no nulos
        rA_arr = np.maximum(rA_arr, 1e-6)
    else:
        rA_arr = k_ev * (CA0_ev * (1 - X_arr))**n

    Lev = FA0_ev / rA_arr

    # Limitar para visualización
    y_max = min(Lev[np.searchsorted(X_arr, X_des)] * 1.7, 200)
    mask   = Lev <= y_max * 1.02
    X_plot = X_arr[mask]
    L_plot = Lev[mask]

    # Encontrar índice de X_des
    i_des = np.searchsorted(X_arr, X_des)
    L_des = Lev[i_des]
    L_des_clipped = min(L_des, y_max)

    # Área PFR (integral numérica)
    X_pfr = X_arr[:i_des+1]
    L_pfr = np.minimum(Lev[:i_des+1], y_max)
    V_PFR_area = np.trapz(L_pfr, X_pfr)

    # Área CSTR (rectángulo)
    V_CSTR_area = L_des_clipped * X_des

    # Relleno PFR (verde)
    ax.fill_between(X_pfr, L_pfr, 0, alpha=0.28, color=COLORS['green'], label=f'V_PFR = {V_PFR_area:.1f} dm³')

    # Relleno CSTR (azul, solo la parte adicional)
    ax.fill_between([0, X_des], [L_des_clipped, L_des_clipped], 0,
                    alpha=0.18, color=COLORS['blue'])
    ax.add_patch(Rectangle((0, 0), X_des, L_des_clipped,
                            fill=False, edgecolor=COLORS['blue'], lw=2,
                            label=f'V_CSTR = {V_CSTR_area:.1f} dm³', linestyle='--'))

    # Curva Levenspiel
    ax.plot(X_plot, L_plot, color=COLORS['gold'], lw=2.5, label='ℒ(X) = F_A0/(-r_A)')

    # Marcadores
    ax.axvline(X_des, color=COLORS['red'], ls=':', lw=1.5)
    ax.scatter([X_des], [L_des_clipped], color=COLORS['gold'], s=100, zorder=6)

    # Anotaciones
    ax.text(X_des/2, L_des_clipped*0.4, 'PFR', ha='center', fontsize=11,
            color=COLORS['green'], fontweight='bold')
    ax.text(X_des*0.55, L_des_clipped*0.85, 'CSTR\nextra', ha='center', fontsize=9,
            color=COLORS['blue'], fontweight='bold')

    ratio = V_CSTR_area / max(V_PFR_area, 0.001)
    ax.set_title(f'{titulo}\nV_CSTR/V_PFR = {ratio:.2f}×', color=COLORS['white'], fontsize=9)
    ax.set_xlabel('Conversión X'); ax.set_ylabel('ℒ(X) = F_A0/(-r_A) [dm³]')
    ax.set_ylim(0, y_max); ax.set_xlim(0, 1)
    ax.legend(fontsize=8, loc='upper left'); ax.grid(True)

plt.tight_layout()
plt.savefig('levenspiel_demo.png', dpi=120, bbox_inches='tight', facecolor='#0A0F1E')
plt.show()
print("✅ Figura guardada: levenspiel_demo.png")


---
# 3. REACTOR PFR — Derivación diferencial rigurosa

## 3.1 Suposiciones del PFR ideal

| # | Suposición | Consecuencia matemática | Cuándo se viola |
|---|------------|------------------------|-----------------|
| **P1** | **Sin mezcla axial** — flujo pistón perfecto | El perfil de concentración solo depende de la posición axial $z$ | Pe_axial < 100, reactores de lecho empacado en transición |
| **P2** | **Mezcla radial perfecta** — sin gradientes radiales | $C_A$, $T$ son solo funciones de $z$ (o $V$) | Tubos anchos sin agitación ($D > 5$ cm para gases) |
| **P3** | **Estado estacionario** | $\partial C_A/\partial t = 0$ en todo punto | Igual que CSTR |
| **P4** | **Sin resistencias de transferencia** | La cinética medida es la cinética intrínseca | Catalizadores con poros grandes, fase gas a baja velocidad |

## 3.2 Derivación diferencial — todos los pasos

**Balance molar sobre un elemento diferencial** $dV$ en estado estacionario (P3):

$$F_A\big|_V - F_A\big|_{V+dV} + r_A\,dV = 0$$

Dividir por $dV$ y tomar el límite $dV \to 0$:

$$\lim_{dV \to 0} \frac{F_A\big|_V - F_A\big|_{V+dV}}{dV} + r_A = 0 \quad \Rightarrow \quad -\frac{dF_A}{dV} = -r_A$$

Usando $F_A = F_{A0}(1-X)$ con $F_{A0}$ constante en estado estacionario:

$$F_{A0}\frac{dX}{dV} = -r_A(X,T,P) \quad \textbf{(Fogler Ec. 2-15)}$$

Integrar de $V=0$ ($X=0$) a $V_\text{PFR}$ ($X=X_A$):

$$\boxed{V_\text{PFR} = F_{A0}\int_0^{X_A}\frac{dX}{-r_A(X)}}$$

## 3.3 Comparación formal CSTR vs PFR

Para cinética $-r_A = kC_{A0}^n(1-X)^n$ con $n>0$, la función integrando es $\mathcal{L}(X) = \frac{F_{A0}}{kC_{A0}^n(1-X)^n}$ — **creciente en X**.

Por el **Teorema del Valor Medio del Cálculo Integral**:
$$\exists\; \bar{X} \in [0, X_A]:\quad V_\text{PFR} = \mathcal{L}(\bar{X})\cdot X_A < \mathcal{L}(X_A)\cdot X_A = V_\text{CSTR}$$

La igualdad se daría solo si $\mathcal{L}$ fuera constante, es decir $n=0$ (orden cero).

## 3.4 Soluciones analíticas para órdenes enteros

| Orden $n$ | $-r_A$ | $\tau_\text{PFR}$ |
|-----------|--------|-------------------|
| 0 | $k$ | $C_{A0}X/k$ |
| 1 | $kC_{A0}(1-X)$ | $-\ln(1-X)/k$ |
| 2 | $kC_{A0}^2(1-X)^2$ | $X/[kC_{A0}(1-X)]$ |
| n | $kC_{A0}^n(1-X)^n$ | $\dfrac{(1-X)^{1-n}-1}{k\,C_{A0}^{n-1}(n-1)}$ para $n\neq1$ |


In [ ]:
# ═══════════════════════════════════════════════════════════
# VERIFICACIÓN NUMÉRICA — PFR para n = 0, 1, 2 y numérico
# Verificar que la integración numérica == solución analítica
# ═══════════════════════════════════════════════════════════

k_pfr, CA0_pfr, v0_pfr = 0.5, 2.0, 10.0
FA0_pfr = CA0_pfr * v0_pfr
X_vals  = np.array([0.40, 0.60, 0.80, 0.90, 0.95])

print("╔══════════════════════════════════════════════════════════════════╗")
print("║  VERIFICACIÓN PFR — Solución analítica vs integración numérica  ║")
print("╠══════════════════════════════════════════════════════════════════╣")

for n, label, tau_analitica_fn in [
    (1, '1er orden', lambda X: -np.log(1-X)/k_pfr),
    (2, '2do orden', lambda X: X/(k_pfr*CA0_pfr*(1-X))),
]:
    print(f"\n  Orden {label}:")
    print(f"  {'X':>6}  {'τ_analítico':>14}  {'τ_numérico':>12}  {'Error %':>10}")
    print(f"  {'-'*50}")
    for X in X_vals:
        # Analítico
        tau_a = tau_analitica_fn(X)
        # Numérico (integración de Gauss)
        integrand = lambda Xv: CA0_pfr / (k_pfr * (CA0_pfr*(1-Xv))**n)
        tau_n, err = quad(integrand, 0, X, limit=100)
        error = abs(tau_a - tau_n)/tau_a * 100
        ok = '✓' if error < 0.01 else '✗'
        print(f"  {X:6.2f}  {tau_a:14.6f}  {tau_n:12.6f}  {error:9.6f}% {ok}")

# ── Figura de perfiles X(V) para distintos n ────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('PFR — Perfiles de conversión para distintos órdenes de reacción',
             color=COLORS['white'], fontweight='bold')

V_range = np.linspace(0, 500, 1000)
tau_range = V_range / v0_pfr

ordenes = [(1, COLORS['gold'], '1er orden'), (2, COLORS['blue'], '2do orden'),
           (0.5, COLORS['green'], '1/2 orden'), (3, COLORS['orange'], '3er orden')]

# Panel 1: X(V) para distintos n
ax = axes[0]
for n, col, lbl in ordenes:
    if n == 1:
        X_pfr = 1 - np.exp(-k_pfr * tau_range)
    elif n == 2:
        X_pfr = k_pfr*CA0_pfr*tau_range / (1 + k_pfr*CA0_pfr*tau_range)
    else:
        X_pfr = 1 - (1 + (n-1)*k_pfr*CA0_pfr**(n-1)*tau_range)**(1/(1-n))
        X_pfr = np.clip(X_pfr, 0, 0.9999)
    ax.plot(V_range, X_pfr*100, color=col, lw=2.3, label=lbl)

ax.set_xlabel('Volumen PFR (dm³)'); ax.set_ylabel('Conversión X (%)')
ax.set_title('Perfil X(V) — distintos órdenes n')
ax.legend(); ax.grid(True); ax.set_ylim(0, 102)

# Panel 2: Razón V_CSTR/V_PFR vs X y n
ax2 = axes[1]
X_r = np.linspace(0.01, 0.95, 400)
for n, col, lbl in ordenes:
    # PFR: integración numérica
    ratios = []
    for xd in X_r:
        V_pfr_num, _ = quad(lambda xv: CA0_pfr/(k_pfr*(CA0_pfr*(1-xv))**n), 0, xd)
        V_pfr_num *= FA0_pfr / CA0_pfr
        V_cstr_num = FA0_pfr * xd / (k_pfr * (CA0_pfr*(1-xd))**n)
        ratios.append(V_cstr_num / max(V_pfr_num, 1e-9))
    ax2.plot(X_r, ratios, color=col, lw=2.3, label=lbl)

ax2.axhline(1, color=COLORS['gray'], ls='--', lw=1.5, label='CSTR = PFR (n=0)')
ax2.set_xlabel('Conversión X'); ax2.set_ylabel('V_CSTR / V_PFR')
ax2.set_title('Penalización del CSTR vs n\n(siempre ≥ 1 para n > 0)')
ax2.legend(); ax2.grid(True); ax2.set_ylim(0.9, 15)
ax2.set_xlim(0, 1)

plt.tight_layout()
plt.savefig('pfr_verificacion.png', dpi=120, bbox_inches='tight', facecolor='#0A0F1E')
plt.show()
print("\n✅ Figura guardada: pfr_verificacion.png")


---
# 4. REACTOR BATCH — Derivación y equivalencia matemática con el PFR

## 4.1 Derivación desde primeros principios

El reactor batch es un **sistema cerrado** — sin flujo de entrada ni salida durante la reacción.
El balance molar para la especie A en el sistema cerrado (Fogler Ec. 1-5):

$$\frac{dN_A}{dt} = r_A V \quad \text{(sin términos de flujo)}$$

Usando $N_A = N_{A0}(1-X)$ con $N_{A0}$ constante:

$$N_{A0}\frac{dX}{dt} = -r_A V$$

Para volumen constante ($V = N_{A0}/C_{A0}$):

$$C_{A0}\frac{dX}{dt} = -r_A \quad \Rightarrow \quad dt = \frac{C_{A0}\,dX}{-r_A}$$

Integrando de $t=0$ ($X=0$) a $t = t_\text{rxn}$ ($X = X_A$):

$$\boxed{t_\text{rxn} = C_{A0}\int_0^{X_A}\frac{dX}{-r_A(X)}} \quad \textbf{(Fogler Ec. 4-5)}$$

## 4.2 Demostración formal de la equivalencia Batch $\equiv$ PFR

Comparando con el PFR:

$$\tau_\text{PFR} = \frac{V_\text{PFR}}{v_0} = \frac{F_{A0}}{v_0}\int_0^X\frac{dX}{-r_A} = C_{A0}\int_0^X\frac{dX}{-r_A}$$

**Por tanto:** $t_\text{batch} = \tau_\text{PFR}$ — **equivalencia matemática exacta**.

La sustitución de variables $\{V \leftrightarrow t,\; F_{A0} \leftrightarrow N_{A0},\; v_0 \leftrightarrow V_R\}$ es biyectiva.

**Consecuencias operacionales:**

- Los datos cinéticos medidos en batch se transfieren directamente al diseño del PFR continuo.
- $t_\text{batch} = \tau_\text{PFR} < \tau_\text{CSTR}$ siempre que $n > 0$.
- El batch es "la planta piloto natural" del PFR.

## 4.3 Optimización de la productividad — X* óptimo

El tiempo de ciclo completo incluye el tiempo inactivo $t_0$:

$$t_\text{ciclo}(X) = t_\text{rxn}(X) + t_0 = C_{A0}\int_0^X\frac{dX}{-r_A} + t_0$$

La **productividad** [mol/(dm³·min)] se maximiza respecto a $X$:

$$\mathcal{P}(X) = \frac{C_{A0}X}{t_\text{rxn}(X) + t_0} \quad \Rightarrow \quad \frac{d\mathcal{P}}{dX} = 0$$

Condición de máximo (derivar e igualar a cero):

$$\frac{C_{A0}[t_\text{ciclo} - X\,\frac{dt_\text{rxn}}{dX}]}{t_\text{ciclo}^2} = 0 \quad \Rightarrow \quad t_\text{ciclo}(X^*) = X^* \cdot \frac{dt_\text{rxn}}{dX}\bigg|_{X^*}$$

Para 1er orden: $\dfrac{dt_\text{rxn}}{dX} = \dfrac{1}{k(1-X)}$, entonces la condición es:

$$\frac{-\ln(1-X^*)}{k} + t_0 = \frac{X^*}{k(1-X^*)} \quad \text{(implícita — solución numérica con brentq)}$$


In [ ]:
# ═══════════════════════════════════════════════════════════
# VERIFICACIÓN BATCH — Equivalencia con PFR y X* óptimo
# ═══════════════════════════════════════════════════════════

k_bt, CA0_bt, t0 = 0.311, 16.1, 45.0   # min⁻¹, mol/dm³, min (Fogler Ex. 4-1)

# Función t_rxn(X) — analítica (1er orden) y numérica
def t_rxn_analitico(X): return -np.log(1 - X) / k_bt
def t_rxn_numerico(X):
    val, _ = quad(lambda Xv: CA0_bt / (k_bt * CA0_bt * (1 - Xv)), 0, X)
    return val

# Productividad
def productividad(X):
    t = t_rxn_analitico(X) + t0
    return CA0_bt * X / t

# Encontrar X* óptimo
def condicion_optimo(X):
    t_rxn = t_rxn_analitico(X)
    dt_rxn_dX = 1 / (k_bt * (1 - X))
    t_ciclo = t_rxn + t0
    return t_ciclo - X * dt_rxn_dX

X_star = brentq(condicion_optimo, 0.01, 0.999)
P_max  = productividad(X_star)

# Comparar analítico vs numérico
X_test = [0.5, 0.7, 0.8, 0.9, 0.95]
print("╔════════════════════════════════════════════════════════╗")
print("║  VERIFICACIÓN BATCH — Analítico vs Numérico           ║")
print("╠════════════════════════════════════════════════════════╣")
print(f"  {'X':>6}  {'t_rxn analítico':>18}  {'t_rxn numérico':>16}  {'Error':>8}")
for X in X_test:
    ta = t_rxn_analitico(X)
    tn = t_rxn_numerico(X)
    print(f"  {X:6.2f}  {ta:18.4f}  {tn:16.4f}  {abs(ta-tn)/ta*100:7.5f}%")

print(f"\n  X* óptimo de productividad = {X_star:.4f}")
print(f"  P_max                      = {P_max:.5f} mol/(dm³·min)")
print(f"  t_rxn en X*                = {t_rxn_analitico(X_star):.2f} min")
print(f"  t_ciclo en X*              = {t_rxn_analitico(X_star)+t0:.2f} min")
print("╚════════════════════════════════════════════════════════╝")

# ── Figura: X(t) y Productividad P(X) ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Batch — Verificación numérica y optimización de productividad',
             color=COLORS['white'], fontweight='bold')

t_arr = np.linspace(0, 200, 500)
X_arr = 1 - np.exp(-k_bt * t_arr)

ax = axes[0]
ax.plot(t_arr, X_arr*100, color=COLORS['gold'], lw=2.5, label='X(t) = 1 − e^{−kt}')
ax.axvline(t_rxn_analitico(X_star), color=COLORS['red'], ls='--', lw=1.8,
           label=f't* = {t_rxn_analitico(X_star):.1f} min')
ax.axhline(X_star*100, color=COLORS['blue'], ls=':', lw=1.8,
           label=f'X* = {X_star:.3f}')
ax.set_xlabel('Tiempo (min)'); ax.set_ylabel('X (%)')
ax.set_title('Perfil X(t) — 1er orden'); ax.legend(); ax.grid(True)

X_range = np.linspace(0.01, 0.999, 500)
P_range = productividad(X_range)

ax2 = axes[1]
ax2.plot(X_range, P_range, color=COLORS['teal'], lw=2.5, label='P(X)')
ax2.axvline(X_star, color=COLORS['red'], ls='--', lw=1.8, label=f'X* = {X_star:.3f}')
ax2.axhline(P_max, color=COLORS['gold'], ls=':', lw=1.8, label=f'P_max = {P_max:.4f}')
ax2.scatter([X_star], [P_max], color=COLORS['gold'], s=150, zorder=6)
ax2.set_xlabel('Conversión X'); ax2.set_ylabel('Productividad P (mol/dm³/min)')
ax2.set_title(f'Productividad P(X)\nt₀ = {t0} min → X* = {X_star:.3f}')
ax2.legend(); ax2.grid(True)

plt.tight_layout()
plt.savefig('batch_verificacion.png', dpi=120, bbox_inches='tight', facecolor='#0A0F1E')
plt.show()
print("✅ Figura guardada: batch_verificacion.png")


---
# 5. BALANCE DE ENERGÍA — Derivación desde el Primer Principio de la Termodinámica

## 5.1 Punto de partida: 1er Principio para sistema abierto en estado estacionario

Para un volumen de control con **un flujo de entrada y uno de salida** en estado estacionario
(energía cinética y potencial despreciables, sin trabajo de eje), el 1er Principio establece
**(Fogler Ec. 8-9)**:

$$\dot{Q} - \dot{W}_s + \sum_i F_{i0}\hat{H}_{i0} - \sum_i F_i\hat{H}_i = 0$$

## 5.2 Expansión de entalpías y simplificación

Usando entalpías de formación a $T_R$ más calor sensible con $C_{p,i}$ constantes:

$$\hat{H}_i(T) = \hat{H}_i^\circ(T_R) + C_{p,i}(T - T_R)$$

Procesando los términos de flujo:
$$\sum_i F_{i0}\hat{H}_{i0} - \sum_i F_i\hat{H}_i = F_{A0}\left[\sum_i\Theta_i C_{p,i}(T_0-T) + (-\Delta H_{rx}(T_R))\,X\right]$$

Con $\dot{Q} = UA(T_a - T)$ para la camisa:

$$\boxed{UA(T_a-T) + F_{A0}\left[\sum_i\Theta_i C_{p,i}(T_0-T) + (-\Delta H_{rx})\,X\right] = 0} \quad \textbf{(Ec. 8-28)}$$

## 5.3 Las dos curvas X_BE(T) y X_BM(T)

**Balance de Energía (BE)** — despejando X:
$$X_{BE}(T) = \frac{\left(\sum_i\Theta_i C_{p,i} + \frac{UA}{F_{A0}}\right)(T-T_0) + \frac{UA}{F_{A0}}(T_0-T_a)}{(-\Delta H_{rx})} \quad \text{[lineal en T]}$$

**Balance de Masa (BM)** — de la ecuación de diseño del CSTR con Arrhenius:
$$X_{BM}(T) = \frac{k_0 e^{-E_a/RT}\,\tau}{1 + k_0 e^{-E_a/RT}\,\tau} \quad \text{[sigmoide en T]}$$

Los **estados estacionarios** son las raíces de $X_{BE}(T) = X_{BM}(T)$.

## 5.4 Demostración del criterio de Van Heerden (1953)

Reformulando como $G(T) = R(T)$:

$$G(T) \equiv (-\Delta H_{rx})\,X_{BM}(T) \quad \text{(calor generado)}$$
$$R(T) \equiv \left(\sum\Theta_i C_{p,i} + \frac{UA}{F_{A0}}\right)(T - T_0^*) \quad \text{(calor removido — recta)}$$

**Demostración de estabilidad:** Para una perturbación $\delta T > 0$ alrededor del SS:
- Si $dR/dT > dG/dT$: el sistema genera calor pero lo remueve más rápido → $dT/dt < 0$ → retorno al SS → **estable**
- Si $dR/dT < dG/dT$: el sistema remueve menos calor del que genera → $dT/dt > 0$ → fuga del SS → **inestable**

$$\boxed{\text{SS estable} \iff \frac{dR}{dT}\bigg|_{T_{ss}} > \frac{dG}{dT}\bigg|_{T_{ss}}}$$


In [ ]:
# ═══════════════════════════════════════════════════════════
# VERIFICACIÓN BALANCE DE ENERGÍA — Multiplicidad CSTR
# Fogler Example 8-4: Hidratación de Óxido de Propileno
# (unidades BTU, lbmol, R)
# ═══════════════════════════════════════════════════════════

# Parámetros del ejemplo
k0      = 16.96e12      # h⁻¹
Ea      = 32400         # BTU/lbmol
R_gas   = 1.987         # BTU/(lbmol·R)
dHrx    = -36400        # BTU/lbmol
sumTCp  = 403           # BTU/(lbmol·R)
CA0     = 0.923         # lbmol/ft³
v0      = 46.2          # ft³/h — para dar FA0
FA0_e   = CA0 * v0      # lbmol/h
UA      = 35500         # BTU/(h·R)
T0      = 528           # R (68°F)
Ta      = T0            # camisa = alimentación (caso simplificado)
tau_e   = 1.0           # h

# Funciones
def k_arr(T):  return k0 * np.exp(-Ea / (R_gas * T))
def X_BM(T):   return k_arr(T)*tau_e / (1 + k_arr(T)*tau_e)
def G(T):      return (-dHrx) * X_BM(T)
def R(T):      return (sumTCp + UA/FA0_e) * (T - T0)

# Encontrar estados estacionarios: G(T) = R(T) → G-R = 0
T_range = np.linspace(510, 710, 4000)
diff    = G(T_range) - R(T_range)

SS_T = []
for i in range(len(diff)-1):
    if diff[i] * diff[i+1] < 0:
        T_ss = brentq(lambda T: G(T)-R(T), T_range[i], T_range[i+1], xtol=1e-8)
        SS_T.append(T_ss)

print("╔════════════════════════════════════════════════════════════╗")
print("║  ESTADOS ESTACIONARIOS — CSTR No Isotérmico (Ex. 8-4)    ║")
print("╠════════════════════════════════════════════════════════════╣")
labels_ss = ['SS Bajo', 'SS Medio', 'SS Alto']
for i, T_ss in enumerate(SS_T):
    X_ss   = X_BM(T_ss)
    dGdT   = (-dHrx) * (k_arr(T_ss)*tau_e * Ea/(R_gas*T_ss**2)) / (1+k_arr(T_ss)*tau_e)**2
    dRdT   = sumTCp + UA/FA0_e
    stable = '✓ ESTABLE' if dRdT > dGdT else '✗ INESTABLE'
    print(f"║  {labels_ss[i]:10s}: T={T_ss:.1f} R  X={X_ss:.4f}  dG/dT={dGdT:.1f}  dR/dT={dRdT:.1f}  {stable} ║")
print("╚════════════════════════════════════════════════════════════╝")

# ── Figura G(T) / R(T) ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Balance de Energía CSTR — Multiplicidad de estados estacionarios\n'
             'Fogler Example 8-4: Hidratación de Óxido de Propileno',
             color=COLORS['white'], fontweight='bold')

ax = axes[0]
G_arr = G(T_range); R_arr = R(T_range)
ax.plot(T_range, G_arr, color=COLORS['orange'], lw=2.5, label='G(T) — calor generado')
ax.plot(T_range, R_arr, color=COLORS['blue'],   lw=2.5, label='R(T) — calor removido')
colors_ss = [COLORS['green'], COLORS['red'], COLORS['green']]
for T_ss, col, lbl in zip(SS_T, colors_ss, labels_ss):
    ax.scatter([T_ss], [G(T_ss)], color=col, s=180, zorder=6)
    ax.annotate(f'{lbl}\nT={T_ss:.0f}R\nX={X_BM(T_ss):.2f}',
                (T_ss, G(T_ss)), xytext=(T_ss+8, G(T_ss)+600),
                fontsize=8, color=col,
                arrowprops=dict(arrowstyle='->', color=col, lw=1.2))
ax.set_xlabel('Temperatura T (R)'); ax.set_ylabel('Calor (BTU/lbmol)')
ax.set_title('Curvas G(T) vs R(T) — Van Heerden')
ax.legend(); ax.grid(True)
ax.set_xlim(510, 700); ax.set_ylim(-5000, 38000)

# Panel derecho — X_BM y X_BE
ax2 = axes[1]
X_BM_arr = X_BM(T_range)
X_BE_arr = (sumTCp + UA/FA0_e)*(T_range - T0) / (-dHrx)
ax2.plot(T_range, X_BM_arr, color=COLORS['orange'], lw=2.5, label='X_BM(T) — balance de masa')
ax2.plot(T_range, X_BE_arr, color=COLORS['blue'],   lw=2.5, label='X_BE(T) — balance de energía')
for T_ss, col, lbl in zip(SS_T, colors_ss, labels_ss):
    ax2.scatter([T_ss], [X_BM(T_ss)], color=col, s=180, zorder=6, label=lbl)
ax2.set_xlabel('Temperatura T (R)'); ax2.set_ylabel('Conversión X')
ax2.set_title('X_BM(T) ∩ X_BE(T) — estados estacionarios')
ax2.legend(fontsize=8); ax2.grid(True)
ax2.set_xlim(510, 700); ax2.set_ylim(-0.05, 1.1)

plt.tight_layout()
plt.savefig('energia_verificacion.png', dpi=120, bbox_inches='tight', facecolor='#0A0F1E')
plt.show()
print("\n✅ Figura guardada: energia_verificacion.png")


---
# 6. CASCADA DE N CSTR — Demostración del límite al PFR

## 6.1 Derivación: N CSTRs iguales en serie (cinética 1er orden)

Para $N$ CSTRs iguales en serie, cada uno con $\tau_i = \tau_\text{total}/N$ y $k$ constante,
el balance de masa del reactor $i$ (entrada $C_{A,i-1}$, salida $C_{A,i}$):

$$C_{A,i-1} - C_{A,i} = k\,\tau_i\,C_{A,i} \quad \Rightarrow \quad C_{A,i} = \frac{C_{A,i-1}}{1+k\tau_i}$$

Aplicando recursivamente desde $C_{A,0} = C_{A0}$:

$$C_{A,N} = \frac{C_{A0}}{(1+k\tau_i)^N} = \frac{C_{A0}}{\left(1+\dfrac{k\tau}{N}\right)^N}$$

**Conversión después de N reactores:**

$$\boxed{X_N = 1 - \frac{1}{\left(1+\dfrac{k\tau}{N}\right)^N}}$$

## 6.2 Demostración formal del límite $N \to \infty$

Usando la definición del número $e$ como límite:

$$\lim_{N \to \infty}\left(1 + \frac{k\tau}{N}\right)^N = e^{k\tau}$$

Por tanto:

$$\lim_{N \to \infty} X_N = 1 - e^{-k\tau} = X_\text{PFR} \quad \textbf{Q.E.D.} \quad \square$$

**La cascada infinita de CSTRs microscópicos ES el PFR.** Geométricamente, los $N$ rectángulos
del diagrama de escalera de Levenspiel convergen al área bajo la curva cuando $N \to \infty$.

## 6.3 Tasa de convergencia

El error de la cascada respecto al PFR decae como $O(1/N)$:

$$X_\text{PFR} - X_N \approx \frac{(k\tau)^2}{2N} e^{-k\tau} + O(1/N^2)$$

Para alcanzar el 95% del rendimiento del PFR se necesita $N \approx 3\text{–}5$ reactores.


In [ ]:
# ═══════════════════════════════════════════════════════════
# VERIFICACIÓN CASCADA — Convergencia al PFR cuando N→∞
# ═══════════════════════════════════════════════════════════

k_cas, tau_cas = 0.5, 4.0   # min⁻¹ y min
X_PFR = 1 - np.exp(-k_cas * tau_cas)

print("╔══════════════════════════════════════════════════════════════╗")
print("║  CONVERGENCIA Cascada N CSTR → PFR (k=0.5, kτ=2.0)        ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║  X_PFR (referencia) = {X_PFR:.8f}                         ║")
print(f"  {'N':>5}  {'X_N':>14}  {'Error abs':>12}  {'% del PFR':>10}  {'V_N/V_PFR':>10}")
print(f"  {'-'*60}")

N_values = [1, 2, 3, 4, 5, 6, 8, 10, 20, 50, 100, 1000]
resultados = []
for N in N_values:
    tau_i = tau_cas / N
    X_N   = 1 - 1/(1 + k_cas*tau_i)**N
    error = abs(X_PFR - X_N)
    pct   = X_N/X_PFR * 100
    # Razón de volúmenes (mismo τ total)
    ratio = N * (X_N/N) / (X_PFR) if X_PFR > 0 else 1  # simplificado
    resultados.append((N, X_N, error, pct))
    print(f"  {N:5d}  {X_N:14.8f}  {error:12.8f}  {pct:9.4f}%")

print(f"\n  Límite N→∞: {X_PFR:.8f} = X_PFR  ✓")
print("╚══════════════════════════════════════════════════════════════╝")

# ── Figura: diagrama de escalera y convergencia ───────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Cascada de CSTRs → PFR cuando N → ∞\nDemostración numérica del límite',
             color=COLORS['white'], fontweight='bold')

# Panel 1: Diagrama de escalera para N = 1, 3, 8
X_lev = np.linspace(0.001, 0.97, 400)
rA_lev = k_cas * 2.0 * (1 - X_lev)   # CA0 = 2
Lev_arr = 10.0 * 2.0 / rA_lev         # FA0 = v0*CA0

ax = axes[0]
ax.plot(X_lev, Lev_arr, color=COLORS['gold'], lw=2.5, label='ℒ(X) = F_A0/(-r_A)')
ax.fill_between(X_lev[X_lev < tau_cas*k_cas/(1+tau_cas*k_cas)],
                Lev_arr[X_lev < tau_cas*k_cas/(1+tau_cas*k_cas)],
                0, alpha=0.22, color=COLORS['green'], label=f'V_PFR (N=∞)')

N_diag = [1, 3]
line_styles = ['--', ':']
col_diag = [COLORS['red'], COLORS['purple']]
for N_d, ls, col_d in zip(N_diag, line_styles, col_diag):
    tau_i_d = tau_cas / N_d
    X_prev = 0
    for j in range(N_d):
        X_next = 1 - 1/(1 + k_cas*tau_i_d)**(j+1)
        i_x = np.searchsorted(X_lev, X_next)
        L_next = Lev_arr[min(i_x, len(Lev_arr)-1)]
        ax.plot([X_prev, X_next], [L_next, L_next], color=col_d, lw=2, ls=ls)
        ax.plot([X_next, X_next], [0, L_next], color=col_d, lw=2, ls=ls)
        X_prev = X_next
    ax.plot([], [], color=col_d, lw=2, ls=ls, label=f'N={N_d} CSTRs')

ax.set_xlabel('Conversión X'); ax.set_ylabel('ℒ(X) = F_A0/(-r_A)')
ax.set_title('Diagrama de Escalera de Levenspiel')
ax.legend(fontsize=9); ax.grid(True); ax.set_ylim(0)

# Panel 2: Convergencia X_N → X_PFR
N_cont = np.logspace(0, 3, 200)
X_N_cont = 1 - 1/(1 + k_cas*tau_cas/N_cont)**N_cont
error_cont = np.abs(X_PFR - X_N_cont)

ax2 = axes[1]
ax2.loglog(N_cont, error_cont, color=COLORS['orange'], lw=2.5, label='|X_PFR − X_N|')
ax2.loglog(N_cont, (k_cas*tau_cas)**2/(2*N_cont)*np.exp(-k_cas*tau_cas),
           color=COLORS['gray'], lw=1.8, ls='--', label='O(1/N) teórico')
ax2.set_xlabel('Número de reactores N'); ax2.set_ylabel('Error |X_PFR − X_N|')
ax2.set_title('Tasa de convergencia al PFR\n(escala log-log)')
ax2.legend(); ax2.grid(True, which='both', alpha=0.4)

plt.tight_layout()
plt.savefig('cascada_convergencia.png', dpi=120, bbox_inches='tight', facecolor='#0A0F1E')
plt.show()
print("✅ Figura guardada: cascada_convergencia.png")


---
# 7. TABLA MAESTRA — Síntesis de las demostraciones rigurosas

| Reactor | Ecuación de diseño | Tipo matemático | Demostración clave | Verificación numérica |
|---------|-------------------|-----------------|-------------------|----------------------|
| **CSTR** | $V = F_{A0}X/(-r_A\|_\text{sal})$ | Álgebra (lineal en $V$) | 6 pasos explícitos desde balance molar | Example 5-1: error < 0.001% |
| **PFR** | $V = F_{A0}\int_0^X dX/(-r_A)$ | ODE de 1er orden | Derivación diferencial sobre $dV$ | Soluciones analíticas n=0,1,2 verificadas |
| **Batch** | $t = C_{A0}\int_0^X dX/(-r_A)$ | ODE (equiv. PFR) | Equivalencia biyectiva $t\leftrightarrow\tau$ | Error analítico vs numérico < 10⁻⁵ |
| **Cascada** | $X_N = 1-(1+k\tau/N)^{-N}$ | Sucesión numérica | Límite $N\to\infty = e^{k\tau}$ (def. de e) | Convergencia O(1/N) verificada |
| **CSTR BE** | $G(T) = R(T)$ | Ecuación trascendente | 1er Principio para sistema abierto | Example 8-4: 3 SS detectados |

## Jerarquía de generalidad

```
Balance General: FA0 - FA + ∫rA dV = dNA/dt
    ↓ SS + mezcla perfecta      ↓ SS + sin mezcla axial    ↓ sin flujo + V cte
   CSTR: V = FA0·X/(-rA|sal)   PFR: dX/dV = -rA/FA0      Batch: dX/dt = -rA/CA0
          ↓ N reactores                ↓ N→∞
      Cascada CSTR ────────────────→ PFR (límite exacto)
```

> **Conclusión fundamental:** Todas las ecuaciones de diseño de reactores ideales son
> **casos especiales** de la misma ley de conservación de masa, obtenidos por la aplicación
> sistemática de suposiciones físicas identificadas y cuantificadas.

---
*Notebook — Demostración Rigurosa · Programa 740484 · Maestría IQ · Universidad del Valle · 2025*
